In [ ]:
# ============================================================
# BÖLÜM 1: CONFIG & SETUP
# ============================================================

# Drive bağla
from google.colab import drive
drive.mount('/content/drive')
!pip install anthropic google-generativeai -q
# Core modülünü import için path ekle
import sys
import os
sys.path.insert(0, '/content/drive/MyDrive/BitirmeProjesi')

# Kontrol: path eklendi mi?
print("Path eklendi:", '/content/drive/MyDrive/BitirmeProjesi' in sys.path)

# Kontrol: dosya var mı?
utils_path = '/content/drive/MyDrive/BitirmeProjesi/core/faithfulness_utils.py'
print("Dosya mevcut:", os.path.exists(utils_path))

# Tüm fonksiyonları import et
from core.faithfulness_utils import *

# Diğer gerekli kütüphaneler
from google.colab import userdata
from openai import OpenAI
from anthropic import Anthropic
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import torch

# --- Proje Path Ayarları ---
PROJECT_FOLDER = "/content/drive/MyDrive/BitirmeProjesi"
DATA_PATH = f"{PROJECT_FOLDER}/Sorular e-tablo - Base_Questions 489.csv"
CHECKPOINT_PATH = f"{PROJECT_FOLDER}/checkpoints/"
RESULTS_PATH = f"{PROJECT_FOLDER}/results/"

# --- API Keys ---
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except:
    GOOGLE_API_KEY = None
# --- Model Ayarları ---
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
NLI_MODEL = "joeddav/xlm-roberta-large-xnli"

# ============================================
# MODEL SEÇİMİ - Sadece birini aktif et
# ============================================
# LLM_MODEL = "gpt-4o-mini"
# LLM_MODEL = "gpt-4o"
#LLM_MODEL  = "claude-sonnet-4-20250514"
LLM_MODEL = "gemini-2.5-pro"
# LLM_MODEL = "gemini-1.5-flash"
# ============================================

# --- Config'i utils'e gönder ---
set_config(
    vector_w=0.4,
    nli_w=0.6,
    llm_model=LLM_MODEL,
    temp=0,
    max_tok=2048,
    checkpoint_path=CHECKPOINT_PATH
)

print("✓ Config yüklendi")
print(f"  Proje: {PROJECT_FOLDER}")
print(f"  Data: {DATA_PATH}")
print(f"  Model: {LLM_MODEL}")
print(f"  OpenAI API: {'✓' if OPENAI_API_KEY else '✗'}")
print(f"  Anthropic API: {'✓' if ANTHROPIC_API_KEY else '✗'}")
print(f"  Google API: {'✓' if GOOGLE_API_KEY else '✗'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.6/832.6 kB 28.8 MB/s eta 0:00:00
Path eklendi: True
Dosya mevcut: True


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✓ Config yüklendi
  Proje: /content/drive/MyDrive/BitirmeProjesi
  Data: /content/drive/MyDrive/BitirmeProjesi/Sorular e-tablo - Base_Questions 489.csv
  Model: gemini-2.5-pro
  OpenAI API: ✓
  Anthropic API: ✓
  Google API: ✓


In [ ]:
# ============================================================
# BÖLÜM 2: MODELLERİ YÜKLE
# ============================================================

print("► Modeller yükleniyor...")

# GPU kontrolü
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Cihaz: {device.upper()}")

# OpenAI client
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
print(f"  OpenAI client: {'✓' if openai_client else '✗'}")

# Claude client
claude_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None
print(f"  Claude client: {'✓' if claude_client else '✗'}")

# Gemini client
gemini_model = None
if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-2.5-pro')
print(f"  Gemini client: {'✓' if gemini_model else '✗'}")

# Embedding model
print("  ► Embedding model yükleniyor...")
emb_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
print(f"  ✓ {EMBEDDING_MODEL} yüklendi")

# NLI model
print("  ► NLI model yükleniyor...")
nli_pipe = pipeline(
    "zero-shot-classification",
    model=NLI_MODEL,
    device=0 if device == "cuda" else -1
)
print(f"  ✓ {NLI_MODEL} yüklendi")

# Modelleri utils'e gönder
set_models(emb_model, nli_pipe, openai_client, claude_client, gemini_model)

print("\n✓ Tüm modeller hazır!")

► Modeller yükleniyor...
  Cihaz: CUDA
  OpenAI client: ✓
  Claude client: ✓
  Gemini client: ✓
  ► Embedding model yükleniyor...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

  ✓ joeddav/xlm-roberta-large-xnli yüklendi

✓ Tüm modeller hazır!


In [ ]:
# ============================================================
# BÖLÜM 3: VERİYİ YÜKLE
# ============================================================

df = load_and_preprocess_data(DATA_PATH)

print(f"\n✓ Kategoriler: {df['category'].unique().tolist()}")
print(f"✓ Zorluk seviyeleri: {df['difficulty'].unique().tolist()}")

In [ ]:
# ============================================================
# BÖLÜM 4: GENERATION
# ============================================================

from datetime import datetime

# Checkpoint ismi
checkpoint_name = f"generation_{datetime.now().strftime('%Y%m%d_%H%M')}"

print(f"► Checkpoint: {checkpoint_name}")

# Generation başlat
df_with_responses = generate_all_responses(df, checkpoint_name=checkpoint_name)

# CSV olarak kaydet
output_csv_path = f"{CHECKPOINT_PATH}results_{checkpoint_name}.csv"
df_with_responses.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
print(f"\n✓ Kaydedildi: {output_csv_path}")

In [ ]:
# ============================================================
# BÖLÜM 4-ALT: CHECKPOINT'TEN YÜKLEME
# (Generation tekrar etmeden, önceki sonuçları kullanmak için)
# ============================================================

import glob

checkpoint_files = glob.glob(f"{CHECKPOINT_PATH}results_*.csv")

if checkpoint_files:
    latest_file = max(checkpoint_files, key=os.path.getmtime)
    print(f"► Checkpoint bulundu: {latest_file}")

    df_with_responses = pd.read_csv(latest_file)
    df_with_responses['gold_evidence_list'] = df_with_responses['gold_evidence'].apply(
        lambda x: [item.strip() for item in str(x).split('|')]
    )

    print(f"✓ {len(df_with_responses)} satır yüklendi")
else:
    print("⚠️ Checkpoint bulunamadı. Önce generation çalıştırın.")

In [ ]:
# ============================================================
# BÖLÜM 5: EVALUATION (Güvenli versiyon - Gemini 2.5 Pro)
# ============================================================
from tqdm.notebook import tqdm
import numpy as np

df_valid = df_with_responses[
    (df_with_responses['error'].isna()) &
    (df_with_responses['model_reasoning'].notna()) &
    (df_with_responses['model_reasoning'].str.len() > 0)
].copy().reset_index(drop=True)

print(f"Toplam: {len(df_with_responses)}, Geçerli: {len(df_valid)}, Çıkarılan: {len(df_with_responses) - len(df_valid)}")

evaluation_results = []
errors = []

for idx, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc="Evaluating"):
    try:
        reasoning = str(row['model_reasoning'])[:5000]
        evidence_list = row['gold_evidence_list']

        semantic_scores = []
        nli_scores = []

        for evidence in evidence_list:
            evidence = str(evidence).strip()[:2000]
            if not evidence:
                continue

            try:
                embeddings = emb_model.encode([reasoning, evidence])
                from scipy.spatial.distance import cosine
                sem_sim = 1 - cosine(embeddings[0], embeddings[1])
                sem_sim = max(0.0, min(1.0, sem_sim))
            except:
                sem_sim = 0.5
            semantic_scores.append(sem_sim)

            try:
                result = nli_pipe(
                    reasoning,
                    candidate_labels=["entailment", "neutral", "contradiction"],
                    hypothesis_template="{}"
                )
                scores = dict(zip(result["labels"], result["scores"]))
                nli_score = (scores.get("entailment", 0) - scores.get("contradiction", 0) + 1) / 2
            except:
                nli_score = 0.5
            nli_scores.append(nli_score)

        avg_semantic = np.mean(semantic_scores) if semantic_scores else 0.5
        avg_nli = np.mean(nli_scores) if nli_scores else 0.5
        faithfulness = 0.4 * avg_semantic + 0.6 * avg_nli

        model_answer = str(row['model_answer']).strip().lower() if row['model_answer'] else ""
        gold_answer = str(row['gold_answer']).strip().lower() if row['gold_answer'] else ""
        exact_match = model_answer == gold_answer

        confidence = row['model_confidence'] if row['model_confidence'] else 50
        conf_norm = confidence / 100

        epistemic_gap = abs(conf_norm - faithfulness)

        if conf_norm > 0.7 and faithfulness < 0.5:
            hall_risk = "YÜKSEK"
        elif conf_norm > 0.7 and faithfulness < 0.7:
            hall_risk = "ORTA"
        else:
            hall_risk = "DÜŞÜK"

        if epistemic_gap < 0.15:
            cal_type = "İyi Kalibre"
        elif conf_norm > faithfulness:
            cal_type = "Overconfident"
        else:
            cal_type = "Underconfident"

        evaluation_results.append({
            'question_id': row['question_id'],
            'category': row['category'],
            'difficulty': row['difficulty'],
            'semantic_similarity': round(avg_semantic, 4),
            'nli_score': round(avg_nli, 4),
            'faithfulness_score': round(faithfulness, 4),
            'exact_match': exact_match,
            'accuracy_score': 1 if exact_match else 0,
            'confidence_normalized': round(conf_norm, 4),
            'epistemic_gap': round(epistemic_gap, 4),
            'hallucination_risk': hall_risk,
            'calibration_type': cal_type,
            'model_answer': row['model_answer'],
            'gold_answer': row['gold_answer']
        })

    except Exception as e:
        errors.append({'question_id': row['question_id'], 'error': str(e)})

df_evaluated = pd.DataFrame(evaluation_results)
print(f"\n✓ Evaluation tamamlandı: {len(df_evaluated)} soru")
print(f"  Hatalı: {len(errors)}")

print("\n" + "="*60)
print("EVALUATION ÖZET")
print("="*60)
print(f"Exact Accuracy: {df_evaluated['accuracy_score'].mean()*100:.1f}%")
print(f"Faithfulness: {df_evaluated['faithfulness_score'].mean():.3f}")
print(f"Confidence: {df_evaluated['confidence_normalized'].mean()*100:.1f}%")
print(f"Epistemic Gap: {df_evaluated['epistemic_gap'].mean():.3f}")
print(f"\nHallucination Risk:")
print(df_evaluated['hallucination_risk'].value_counts())
print(f"\nCalibration:")
print(df_evaluated['calibration_type'].value_counts())

In [ ]:
# ============================================================
# BÖLÜM 5: EVALUATION
# ============================================================

df_evaluated = run_evaluation(df_with_responses)

# Özet sonuçları göster
print_evaluation_summary(df_evaluated)

# Kaydet
eval_output_path = f"{RESULTS_PATH}evaluation_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df_evaluated.to_csv(eval_output_path, index=False, encoding='utf-8-sig')
print(f"\n✓ Sonuçlar kaydedildi: {eval_output_path}")

In [ ]:
# ============================================================
# BÖLÜM 8: SOFT ACCURACY HESAPLAMA
# ============================================================

from scipy.spatial.distance import cosine
from tqdm.notebook import tqdm

def calculate_soft_accuracy_local(model_answer, gold_answer, semantic_threshold=0.85, nli_threshold=0.4):
    """Hibrit soft accuracy: Exact match veya (Semantic >= 0.85 AND NLI >= 0.4)"""
    if not model_answer or not gold_answer:
        return {"soft_match": False, "accuracy_score": 0, "match_type": "no_answer",
                "semantic_sim": 0.0, "nli_entailment": 0.0}

    model_norm = str(model_answer).strip().lower()
    gold_norm = str(gold_answer).strip().lower()

    # Exact match
    if model_norm == gold_norm:
        return {"soft_match": True, "accuracy_score": 1, "match_type": "exact",
                "semantic_sim": 1.0, "nli_entailment": 1.0}

    # Semantic similarity
    embeddings = emb_model.encode([str(model_answer), str(gold_answer)])
    semantic_sim = 1 - cosine(embeddings[0], embeddings[1])
    semantic_sim = max(0.0, min(1.0, semantic_sim))

    # NLI entailment
    nli_result = nli_pipe(str(model_answer), candidate_labels=["entailment", "neutral", "contradiction"])
    scores = dict(zip(nli_result["labels"], nli_result["scores"]))
    nli_entailment = scores.get("entailment", 0)

    # Soft match
    soft_match = (semantic_sim >= semantic_threshold) and (nli_entailment >= nli_threshold)

    return {"soft_match": soft_match, "accuracy_score": 1 if soft_match else 0,
            "match_type": "soft" if soft_match else "no_match",
            "semantic_sim": round(semantic_sim, 4), "nli_entailment": round(nli_entailment, 4)}

# Soft accuracy hesapla
print("► Soft Accuracy hesaplanıyor...")
soft_results = []
for idx, row in tqdm(df_evaluated.iterrows(), total=len(df_evaluated), desc="Soft Accuracy"):
    result = calculate_soft_accuracy_local(row['model_answer'], row['gold_answer'])
    result['question_id'] = row['question_id']
    soft_results.append(result)

soft_df = pd.DataFrame(soft_results)
df_evaluated['soft_match'] = soft_df['soft_match'].values
df_evaluated['soft_accuracy_score'] = soft_df['accuracy_score'].values
df_evaluated['match_type'] = soft_df['match_type'].values
df_evaluated['answer_semantic_sim'] = soft_df['semantic_sim'].values
df_evaluated['answer_nli_entailment'] = soft_df['nli_entailment'].values

print("✓ Soft Accuracy eklendi!")

# Sonuçları kaydet
eval_soft_path = f"{RESULTS_PATH}evaluation_soft_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df_evaluated.to_csv(eval_soft_path, index=False, encoding='utf-8-sig')
print(f"✓ Kaydedildi: {eval_soft_path}")

In [ ]:
# ============================================================
# BÖLÜM 6: DETAYLI SONUÇLAR
# ============================================================

display(df_evaluated[[
    'question_id', 'category', 'difficulty',
    'accuracy_score', 'faithfulness_score',
    'confidence_normalized', 'epistemic_gap',
    'hallucination_risk', 'calibration_type'
]])

In [ ]:
# ============================================================
# BÖLÜM 9: VISUALIZATION SETUP
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.style.use('seaborn-v0_8-whitegrid')

print("✓ Visualization hazır")

In [ ]:
# ============================================================
# BÖLÜM 10: QUADRANT ANALİZİ
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8))

colors = ['green' if acc == 1 else 'red' for acc in df_evaluated['accuracy_score']]

ax.scatter(df_evaluated['faithfulness_score'], df_evaluated['confidence_normalized'],
           c=colors, s=100, alpha=0.7, edgecolors='black', linewidth=1)

ax.axhline(y=0.7, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(x=0.7, color='gray', linestyle='--', linewidth=1, alpha=0.7)

ax.text(0.35, 0.85, 'YÜKSEK RİSK\n(Halüsinasyon)', fontsize=10, ha='center', color='red', fontweight='bold')
ax.text(0.85, 0.85, 'İDEAL\n(Güvenilir)', fontsize=10, ha='center', color='green', fontweight='bold')
ax.text(0.35, 0.55, 'DÜŞÜK GÜVEN\n(Temkinli)', fontsize=10, ha='center', color='orange')
ax.text(0.85, 0.55, 'AZ GÜVEN\n(Mütevazı)', fontsize=10, ha='center', color='blue')

ax.set_xlabel('Faithfulness Score', fontsize=12)
ax.set_ylabel('Confidence (Normalized)', fontsize=12)
ax.set_title(f'{LLM_MODEL}: Epistemik Güvenilirlik Analizi\n(Yeşil=Doğru, Kırmızı=Yanlış)', fontsize=14)
ax.set_xlim(0.3, 1.0)
ax.set_ylim(0.4, 1.05)

legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=10, label='Doğru'),
                   Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Yanlış')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(f"{RESULTS_PATH}quadrant_{LLM_MODEL.replace('-','')}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# BÖLÜM 11: HALÜSİNASYON VE CALIBRATION
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie Chart
risk_counts = df_evaluated['hallucination_risk'].value_counts()
colors_pie = {'DÜŞÜK': 'green', 'ORTA': 'orange', 'YÜKSEK': 'red'}
pie_colors = [colors_pie.get(risk, 'gray') for risk in risk_counts.index]
axes[0].pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%', colors=pie_colors,
            explode=[0.05 if risk == 'YÜKSEK' else 0 for risk in risk_counts.index], startangle=90)
axes[0].set_title(f'{LLM_MODEL}: Halüsinasyon Riski', fontsize=12)

# Bar Chart
cal_counts = df_evaluated['calibration_type'].value_counts()
colors_bar = {'Overconfident': 'red', 'İyi Kalibre': 'green', 'Underconfident': 'blue'}
bar_colors = [colors_bar.get(cal, 'gray') for cal in cal_counts.index]
axes[1].bar(cal_counts.index, cal_counts.values, color=bar_colors, edgecolor='black')
axes[1].set_title(f'{LLM_MODEL}: Calibration Tipi', fontsize=12)
axes[1].set_ylabel('Soru Sayısı')
for i, (cal, count) in enumerate(cal_counts.items()):
    axes[1].text(i, count + 2, f'{count}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{RESULTS_PATH}hallucination_{LLM_MODEL.replace('-','')}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# BÖLÜM 12: EXACT vs SOFT ACCURACY
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

categories = df_evaluated['category'].unique()
x = range(len(categories))
width = 0.35

exact_accs = [df_evaluated[df_evaluated['category'] == cat]['accuracy_score'].mean()*100 for cat in categories]
soft_accs = [df_evaluated[df_evaluated['category'] == cat]['soft_accuracy_score'].mean()*100 for cat in categories]

bars1 = ax.bar([i - width/2 for i in x], exact_accs, width, label='Exact', color='steelblue')
bars2 = ax.bar([i + width/2 for i in x], soft_accs, width, label='Soft', color='coral')

ax.set_xlabel('Kategori')
ax.set_ylabel('Accuracy (%)')
ax.set_title(f'{LLM_MODEL}: Exact vs Soft Accuracy', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 110)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{bar.get_height():.1f}%', ha='center', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{bar.get_height():.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f"{RESULTS_PATH}exact_vs_soft_{LLM_MODEL.replace('-','')}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# BÖLÜM 13: KATEGORİ VE ZORLUK PERFORMANSI
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
width = 0.35

# Kategori Bazlı
cat_stats = df_evaluated.groupby('category').agg({'soft_accuracy_score': 'mean', 'faithfulness_score': 'mean'}).reset_index()
x = range(len(cat_stats))
bars1 = axes[0].bar([i - width/2 for i in x], cat_stats['soft_accuracy_score']*100, width, label='Soft Acc', color='steelblue')
bars2 = axes[0].bar([i + width/2 for i in x], cat_stats['faithfulness_score']*100, width, label='Faith', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cat_stats['category'], rotation=45, ha='right')
axes[0].set_ylabel('Skor (%)')
axes[0].set_title('Kategori Bazlı')
axes[0].legend()
axes[0].set_ylim(0, 110)

# Zorluk Bazlı
diff_order = ['easy', 'medium', 'hard']
diff_stats = df_evaluated.groupby('difficulty').agg({'soft_accuracy_score': 'mean', 'faithfulness_score': 'mean'}).reindex(diff_order).reset_index()
x2 = range(len(diff_stats))
bars3 = axes[1].bar([i - width/2 for i in x2], diff_stats['soft_accuracy_score']*100, width, label='Soft Acc', color='steelblue')
bars4 = axes[1].bar([i + width/2 for i in x2], diff_stats['faithfulness_score']*100, width, label='Faith', color='coral')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(['Kolay', 'Orta', 'Zor'])
axes[1].set_ylabel('Skor (%)')
axes[1].set_title('Zorluk Bazlı')
axes[1].legend()
axes[1].set_ylim(0, 110)

plt.tight_layout()
plt.savefig(f"{RESULTS_PATH}category_difficulty_{LLM_MODEL.replace('-','')}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# BÖLÜM 14: ÖZET RAPOR
# ============================================================

n = len(df_evaluated)
print("=" * 70)
print(f"        {LLM_MODEL} ({n} soru) - FİNAL ÖZET RAPOR")
print("=" * 70)

print(f"""
GENEL PERFORMANS
────────────────
Exact Accuracy:    {df_evaluated['accuracy_score'].mean()*100:.1f}% ({int(df_evaluated['accuracy_score'].sum())}/{n})
Soft Accuracy:     {df_evaluated['soft_accuracy_score'].mean()*100:.1f}% ({int(df_evaluated['soft_accuracy_score'].sum())}/{n})
Faithfulness:      {df_evaluated['faithfulness_score'].mean():.3f}
Confidence:        {df_evaluated['confidence_normalized'].mean()*100:.1f}%
Epistemic Gap:     {df_evaluated['epistemic_gap'].mean():.3f}

EPİSTEMİK ANALİZ
────────────────
Overconfident:     {(df_evaluated['calibration_type'] == 'Overconfident').sum()} soru ({(df_evaluated['calibration_type'] == 'Overconfident').mean()*100:.1f}%)
İyi Kalibre:       {(df_evaluated['calibration_type'] == 'İyi Kalibre').sum()} soru
Hallucination:     {(df_evaluated['hallucination_risk'] == 'YÜKSEK').sum()} YÜKSEK, {(df_evaluated['hallucination_risk'] == 'ORTA').sum()} ORTA, {(df_evaluated['hallucination_risk'] == 'DÜŞÜK').sum()} DÜŞÜK

KATEGORİ BAZLI (Soft Accuracy)
──────────────────────────────""")

for cat in df_evaluated['category'].unique():
    cat_data = df_evaluated[df_evaluated['category'] == cat]
    print(f"{cat}: {cat_data['soft_accuracy_score'].mean()*100:.1f}% (n={len(cat_data)}, faith={cat_data['faithfulness_score'].mean():.3f})")

print(f"""
ZORLUK BAZLI (Soft Accuracy)
────────────────────────────
Kolay:   {df_evaluated[df_evaluated['difficulty'] == 'easy']['soft_accuracy_score'].mean()*100:.1f}%
Orta:    {df_evaluated[df_evaluated['difficulty'] == 'medium']['soft_accuracy_score'].mean()*100:.1f}%
Zor:     {df_evaluated[df_evaluated['difficulty'] == 'hard']['soft_accuracy_score'].mean()*100:.1f}%
""")
print("=" * 70)

In [ ]:
# ============================================================
# DEMO ARAYÜZÜ - Checkpoint'ten yükle, faithfulness göster
# Sadece bu hücreyi çalıştır, diğer hücrelere dokunma
# ============================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import ast

# --- PATH'i kendi checkpoint dosyana göre güncelle ---
CHECKPOINT_CSV = f"{CHECKPOINT_PATH}results_generation_20260306_1437.csv"
# -----------------------------------------------------

# Veriyi yükle
df_demo = pd.read_csv(CHECKPOINT_CSV, encoding='utf-8-sig')
df_demo = df_demo[df_demo['error'].isna() & df_demo['model_reasoning'].notna()].reset_index(drop=True)

def parse_evidence(ev):
    if isinstance(ev, list):
        return ev
    try:
        return ast.literal_eval(str(ev))
    except:
        return [e.strip() for e in str(ev).split('|') if e.strip()]

df_demo['gold_evidence_list'] = df_demo['gold_evidence'].apply(parse_evidence)

# Faithfulness hesaplama (notebook ile ayni: 0.4 semantic + 0.6 NLI)
def compute_faithfulness_demo(reasoning, evidence_list):
    from scipy.spatial.distance import cosine as scipy_cosine
    semantic_scores = []
    nli_scores = []
    for ev in evidence_list:
        ev = str(ev).strip()[:2000]
        if not ev:
            continue
        try:
            embs = emb_model.encode([reasoning[:5000], ev])
            sem = float(1 - scipy_cosine(embs[0], embs[1]))
            sem = max(0.0, min(1.0, sem))
        except:
            sem = 0.5
        semantic_scores.append(sem)
        try:
            result = nli_pipe(
                reasoning[:5000],
                candidate_labels=["entailment", "neutral", "contradiction"],
                hypothesis_template="{}"
            )
            scores = dict(zip(result["labels"], result["scores"]))
            nli = (scores.get("entailment", 0) - scores.get("contradiction", 0) + 1) / 2
        except:
            nli = 0.5
        nli_scores.append(nli)
    avg_sem = float(np.mean(semantic_scores)) if semantic_scores else 0.5
    avg_nli = float(np.mean(nli_scores)) if nli_scores else 0.5
    faith = 0.4 * avg_sem + 0.6 * avg_nli
    return round(avg_sem, 3), round(avg_nli, 3), round(faith, 3)

# Kalibrasyon ve risk
def get_calibration(conf_norm, faith):
    gap = abs(conf_norm - faith)
    if gap < 0.15:
        return "İyi Kalibre", gap
    elif conf_norm > faith:
        return "Overconfident", gap
    else:
        return "Underconfident", gap

def get_hallucination_risk(conf_norm, faith):
    if conf_norm > 0.7 and faith < 0.5:
        return "YÜKSEK"
    elif conf_norm > 0.7 and faith < 0.7:
        return "ORTA"
    else:
        return "DÜŞÜK"

# Kategori ve zorluk seçenekleri
cats = ["Tümü"] + sorted(df_demo['category'].unique().tolist())
diffs = ["Tümü"] + ["easy", "medium", "hard"]

# Widget'lar
style = {'description_width': '130px'}
layout_wide = widgets.Layout(width='400px')
layout_med = widgets.Layout(width='200px')

w_cat = widgets.Dropdown(options=cats, value="Tümü", description="Kategori:", style=style, layout=layout_wide)
w_diff = widgets.Dropdown(options=diffs, value="Tümü", description="Zorluk:", style=style, layout=layout_med)
w_q = widgets.Dropdown(description="Soru:", style=style, layout=widgets.Layout(width='600px'))
w_btn = widgets.Button(description="Analiz Et", button_style='primary', layout=widgets.Layout(width='160px'))
out = widgets.Output()

def update_questions(change=None):
    cat = w_cat.value
    diff = w_diff.value
    tmp = df_demo.copy()
    if cat != "Tümü":
        tmp = tmp[tmp['category'] == cat]
    if diff != "Tümü":
        tmp = tmp[tmp['difficulty'] == diff]
    opts = [(f"[{r['difficulty']}] {r['question_id']} - {r['question'][:60]}...", i)
            for i, r in tmp.iterrows()]
    w_q.options = opts
    w_q.value = opts[0][1] if opts else None

w_cat.observe(update_questions, names='value')
w_diff.observe(update_questions, names='value')
update_questions()

def on_analyze(b):
    with out:
        clear_output(wait=True)
        idx = w_q.value
        if idx is None:
            print("Soru seçilmedi.")
            return
        row = df_demo.loc[idx]
        reasoning = str(row['model_reasoning'])
        evidence_list = row['gold_evidence_list']
        confidence = float(row['model_confidence']) if pd.notna(row['model_confidence']) else 50.0
        conf_norm = confidence / 100.0

        print("Hesaplanıyor, lütfen bekle...\n")

        avg_sem, avg_nli, faith = compute_faithfulness_demo(reasoning, evidence_list)
        cal_type, gap = get_calibration(conf_norm, faith)
        hall_risk = get_hallucination_risk(conf_norm, faith)
        exact_match = str(row['model_answer']).strip().lower() == str(row['gold_answer']).strip().lower()

        sep = "=" * 65
        print(sep)
        print(f"  {row['question_id']} | {row['category']} | {row['difficulty']}")
        print(sep)

        print("\nSORU")
        print("-" * 40)
        print(row['question'])

        print("\nBAĞLAM")
        print("-" * 40)
        print(row['context'])

        print("\nMODEL REASONING (Chain-of-Thought)")
        print("-" * 40)
        print(reasoning)

        print("\nCEVAPLAR")
        print("-" * 40)
        print(f"  Model cevabı  : {row['model_answer']}")
        print(f"  Gold answer   : {row['gold_answer']}")
        print(f"  Exact match   : {'EVET' if exact_match else 'HAYIR'}")

        print("\nGOLD EVIDENCE")
        print("-" * 40)
        for i, ev in enumerate(evidence_list, 1):
            print(f"  {i}. {ev}")

        print("\nFAITHFULNESS SKORLARI")
        print("-" * 40)
        print(f"  Semantik benzerlik (w=0.4) : {avg_sem:.3f}")
        print(f"  NLI skoru      (w=0.6)     : {avg_nli:.3f}")
        print(f"  Hibrit faithfulness        : {faith:.3f}")

        print("\nEPİSTEMİK KALİBRASYON")
        print("-" * 40)
        print(f"  Güven skoru    : {confidence:.0f}% ({conf_norm:.2f})")
        print(f"  Epistemic gap  : {gap:.3f}")
        print(f"  Kalibrasyon    : {cal_type}")
        print(f"  Halüsinasyon   : {hall_risk}")
        print(sep)

w_btn.on_click(on_analyze)

display(
    widgets.HBox([w_cat, w_diff]),
    w_q,
    w_btn,
    out
)

Dropdown(description='Soru:', layout=Layout(width='600px'), options=(('[easy] NR-1-001 - Buna göre, D sayısı k…

Button(button_style='primary', description='Analiz Et', layout=Layout(width='160px'), style=ButtonStyle())

Output()

In [ ]:
# ============================================================
# DEMO ARAYÜZÜ - Gradio versiyonu
# Bölüm 1 ve 2 çalıştırıldıktan sonra bu hücreyi çalıştır
# ============================================================

!pip install gradio -q

import gradio as gr
import pandas as pd
import numpy as np
import ast
from scipy.spatial.distance import cosine as scipy_cosine

# --- PATH güncelle ---
CHECKPOINT_CSV = f"{CHECKPOINT_PATH}results_generation_20260306_1437.csv"
# ---------------------

df_demo = pd.read_csv(CHECKPOINT_CSV, encoding='utf-8-sig')
df_demo = df_demo[df_demo['error'].isna() & df_demo['model_reasoning'].notna()].reset_index(drop=True)

def parse_evidence(ev):
    try:
        return ast.literal_eval(str(ev))
    except:
        return [e.strip() for e in str(ev).split('|') if e.strip()]

df_demo['gold_evidence_list'] = df_demo['gold_evidence'].apply(parse_evidence)

CAT_LABELS = {
    "numerical_reasoning": "Sayisal",
    "temporal_reasoning": "Zamansal",
    "causal_reasoning": "Nedensel",
    "logical_inference": "Mantiksal Cikarsam",
    "analogical_reasoning": "Analojik",
    "compositional_reasoning": "Bilesik",
}
DIFF_COLORS = {"easy": "easy", "medium": "medium", "hard": "hard"}

def compute_faithfulness(reasoning, evidence_list):
    sem_scores, nli_scores = [], []
    for ev in evidence_list:
        ev = str(ev).strip()[:2000]
        if not ev:
            continue
        try:
            embs = emb_model.encode([reasoning[:5000], ev])
            sem = float(max(0.0, min(1.0, 1 - scipy_cosine(embs[0], embs[1]))))
        except:
            sem = 0.5
        sem_scores.append(sem)
        try:
            res = nli_pipe(reasoning[:5000], candidate_labels=["entailment","neutral","contradiction"], hypothesis_template="{}")
            sc = dict(zip(res["labels"], res["scores"]))
            nli = (sc.get("entailment",0) - sc.get("contradiction",0) + 1) / 2
        except:
            nli = 0.5
        nli_scores.append(nli)
    avg_sem = float(np.mean(sem_scores)) if sem_scores else 0.5
    avg_nli = float(np.mean(nli_scores)) if nli_scores else 0.5
    return round(avg_sem, 3), round(avg_nli, 3), round(0.4*avg_sem + 0.6*avg_nli, 3)

def get_soru_listesi(kategori, zorluk):
    tmp = df_demo.copy()
    if kategori != "Tümü":
        tmp = tmp[tmp['category'] == kategori]
    if zorluk != "Tümü":
        tmp = tmp[tmp['difficulty'] == zorluk]
    choices = [
        f"[{r['difficulty']}] {r['question_id']} - {r['question'][:70]}"
        for _, r in tmp.iterrows()
    ]
    return gr.update(choices=choices, value=choices[0] if choices else None)

def get_row(soru_str):
    if not soru_str:
        return None
    qid = soru_str.split("] ")[1].split(" - ")[0].strip()
    matches = df_demo[df_demo['question_id'] == qid]
    return matches.iloc[0] if not matches.empty else None

def analiz_et(kategori, zorluk, soru_str):
    row = get_row(soru_str)
    if row is None:
        return [gr.update(value="Soru bulunamadi.")] * 9 + [gr.update(visible=False)] * 4

    reasoning = str(row['model_reasoning'])
    evidence_list = row['gold_evidence_list']
    confidence = float(row['model_confidence']) if pd.notna(row['model_confidence']) else 50.0
    conf_norm = confidence / 100.0

    avg_sem, avg_nli, faith = compute_faithfulness(reasoning, evidence_list)
    gap = round(abs(conf_norm - faith), 3)

    if gap < 0.15:
        cal_type, cal_color = "Iyi Kalibre", "green"
    elif conf_norm > faith:
        cal_type, cal_color = "Overconfident", "red"
    else:
        cal_type, cal_color = "Underconfident", "orange"

    if conf_norm > 0.7 and faith < 0.5:
        hall, hall_color = "YÜKSEK", "red"
    elif conf_norm > 0.7 and faith < 0.7:
        hall, hall_color = "ORTA", "orange"
    else:
        hall, hall_color = "DÜŞÜK", "green"

    exact = str(row['model_answer']).strip().lower() == str(row['gold_answer']).strip().lower()
    cat_label = CAT_LABELS.get(row['category'], row['category'])
    evidence_md = "\n".join([f"**{i+1}.** {ev}" for i, ev in enumerate(evidence_list)])

    info_md = f"""### {row['question_id']} - {cat_label} - {row['difficulty']}

**Soru:**
{row['question']}

**Baglam:**
{row['context']}"""

    reasoning_md = f"""### Model Reasoning (Chain-of-Thought)
{reasoning}

---
| | |
|---|---|
| **Model cevabi** | {row['model_answer']} |
| **Gold answer** | {row['gold_answer']} |
| **Exact match** | {"EVET" if exact else "HAYIR"} |"""

    evidence_out = f"### Gold Evidence\n{evidence_md}"

    faith_md = f"""### Faithfulness Skorlari

| Metrik | Agirlik | Skor |
|---|---|---|
| Semantik benzerlik | 0.4 | **{avg_sem:.3f}** |
| NLI skoru (LLM-Judge) | 0.6 | **{avg_nli:.3f}** |
| **Hibrit Faithfulness** | - | **{faith:.3f}** |"""

    cal_md = f"""### Epistemik Kalibrasyon

| | |
|---|---|
| Güven skoru | **{confidence:.0f}%** ({conf_norm:.2f}) |
| Faithfulness | **{faith:.3f}** |
| Epistemic gap | **{gap:.3f}** |
| Kalibrasyon | **{cal_type}** |
| Halüsinasyon riski | **{hall}** |"""

    return info_md, reasoning_md, evidence_out, faith_md, cal_md, faith, avg_sem, avg_nli, confidence/100

cats = ["Tümü"] + sorted(df_demo['category'].unique().tolist())
diffs = ["Tümü", "easy", "medium", "hard"]

with gr.Blocks(title="Türkçe LLM Faithfulness Demo", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# Türkçe LLM Sadakat ve Epistemik Kalibrasyon Demosu")
    gr.Markdown("Veri setinden bir soru seçin - modelin chain-of-thought reasoning adimlarini gold evidence ile karsilastirin.")

    with gr.Row():
        dd_cat  = gr.Dropdown(choices=cats, value="Tümü", label="Kategori", scale=2)
        dd_diff = gr.Dropdown(choices=diffs, value="Tümü", label="Zorluk", scale=1)

    dd_soru = gr.Dropdown(label="Soru", choices=[], scale=4)
    btn = gr.Button("Analiz Et", variant="primary", size="lg")

    with gr.Row():
        with gr.Column(scale=3):
            out_info      = gr.Markdown()
            out_reasoning = gr.Markdown()
            out_evidence  = gr.Markdown()

        with gr.Column(scale=2):
            out_faith_md = gr.Markdown()
            out_cal_md   = gr.Markdown()

            gr.Markdown("### Görsel Özet")
            with gr.Row():
                num_faith = gr.Number(label="Faithfulness", precision=3)
                num_sem   = gr.Number(label="Semantik Benzerlik", precision=3)
            with gr.Row():
                num_nli   = gr.Number(label="NLI Skoru", precision=3)
                num_conf  = gr.Number(label="Güven (norm)", precision=2)

    dd_cat.change(get_soru_listesi, inputs=[dd_cat, dd_diff], outputs=dd_soru)
    dd_diff.change(get_soru_listesi, inputs=[dd_cat, dd_diff], outputs=dd_soru)

    btn.click(
        analiz_et,
        inputs=[dd_cat, dd_diff, dd_soru],
        outputs=[out_info, out_reasoning, out_evidence, out_faith_md, out_cal_md,
                 num_faith, num_sem, num_nli, num_conf]
    )

    demo.load(get_soru_listesi, inputs=[dd_cat, dd_diff], outputs=dd_soru)

demo.launch(share=True, debug=False)

/tmp/ipykernel_3351/2542845469.py:155: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Türkçe LLM Faithfulness Demo", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://200919da994dc66e6f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# DEMO ARAYÜZÜ v2 - Evaluation CSV'den direkt oku
# Sadece bu hücreyi çalıştır, başka hiçbir hücre gerekmez
# ============================================================

!pip install gradio -q

import gradio as gr
import pandas as pd
import ast

# --- Hangi evaluation CSV'lerini göstermek istiyorsan buraya ekle ---
MODEL_FILES = {
    "GPT-4o": "/content/drive/MyDrive/BitirmeProjesi/results/evaluation_soft_20260306_1515gpt4.csv",

    # "Claude Sonnet 4": f"{CHECKPOINT_PATH}../results/evaluation_soft_XXXX_claude.csv",
    # "Gemini 2.5 Flash": f"{CHECKPOINT_PATH}../results/evaluation_soft_XXXX_gemini.csv",
}
# -------------------------------------------------------------------

def load_df(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    df = df[df['model_reasoning'].notna()].reset_index(drop=True)
    def parse_ev(ev):
        try:
            return ast.literal_eval(str(ev))
        except:
            return [e.strip() for e in str(ev).split('|') if e.strip()]
    df['gold_evidence_list'] = df['gold_evidence'].apply(parse_ev)
    return df

DFS = {name: load_df(path) for name, path in MODEL_FILES.items()}

CAT_LABELS = {
    "numerical_reasoning":    "Sayisal",
    "temporal_reasoning":     "Zamansal",
    "causal_reasoning":       "Nedensel",
    "logical_inference":      "Mantiksal Cikarsam",
    "analogical_reasoning":   "Analojik",
    "compositional_reasoning":"Bilesik",
}

def get_soru_listesi(model, kategori, zorluk):
    df = DFS.get(model, list(DFS.values())[0])
    tmp = df.copy()
    if kategori != "Tümü":
        tmp = tmp[tmp['category'] == kategori]
    if zorluk != "Tümü":
        tmp = tmp[tmp['difficulty'] == zorluk]
    choices = [
        f"[{r['difficulty']}] {r['question_id']} - {r['question'][:70]}"
        for _, r in tmp.iterrows()
    ]
    return gr.update(choices=choices, value=choices[0] if choices else None)

def analiz_et(model, kategori, zorluk, soru_str):
    if not soru_str:
        return ["Soru seçilmedi."] * 5 + [0.0] * 4

    df = DFS.get(model, list(DFS.values())[0])
    qid = soru_str.split("] ")[1].split(" - ")[0].strip()
    matches = df[df['question_id'] == qid]
    if matches.empty:
        return ["Soru bulunamadi."] * 5 + [0.0] * 4
    row = matches.iloc[0]

    # Skorlari direkt CSV'den oku
    faith     = float(row['faithfulness_score'])
    avg_sem   = float(row['semantic_similarity'])
    avg_nli   = float(row['nli_score'])
    conf_norm = float(row['confidence_normalized'])
    gap       = float(row['epistemic_gap'])
    cal_type  = str(row['calibration_type'])
    hall      = str(row['hallucination_risk'])
    exact     = bool(row['exact_match'])
    soft      = bool(row.get('soft_match', False))

    cat_label   = CAT_LABELS.get(row['category'], row['category'])
    evidence_md = "\n".join([f"**{i+1}.** {ev}" for i, ev in enumerate(row['gold_evidence_list'])])

    info_md = f"""### {row['question_id']} - {cat_label} - {row['difficulty']}

**Soru:**
{row['question']}

**Baglam:**
{row['context']}"""

    reasoning_md = f"""### Model Reasoning - {model}
{row['model_reasoning']}

---
| | |
|---|---|
| **Model cevabi** | {row['model_answer']} |
| **Gold answer**  | {row['gold_answer']} |
| **Exact match**  | {"EVET" if exact else "HAYIR"} |
| **Soft match**   | {"EVET" if soft else "HAYIR"} |"""

    evidence_out = f"### Gold Evidence\n{evidence_md}"

    faith_md = f"""### Faithfulness Skorlari

| Metrik | Agirlik | Skor |
|---|---|---|
| Semantik benzerlik | 0.4 | **{avg_sem:.3f}** |
| NLI skoru | 0.6 | **{avg_nli:.3f}** |
| **Hibrit Faithfulness** | - | **{faith:.3f}** |"""

    cal_emoji = {"İyi Kalibre": "Iyi Kalibre", "Overconfident": "Overconfident", "Underconfident": "Underconfident"}.get(cal_type, cal_type)
    hall_emoji = {"YÜKSEK": "YÜKSEK", "ORTA": "ORTA", "DÜŞÜK": "DÜŞÜK"}.get(hall, hall)

    cal_md = f"""### Epistemik Kalibrasyon

| | |
|---|---|
| Model | **{model}** |
| Güven skoru | **{conf_norm*100:.0f}%** ({conf_norm:.2f}) |
| Faithfulness | **{faith:.3f}** |
| Epistemic gap | **{gap:.3f}** |
| Kalibrasyon | **{cal_emoji}** |
| Halüsinasyon riski | **{hall_emoji}** |"""

    return info_md, reasoning_md, evidence_out, faith_md, cal_md, faith, avg_sem, avg_nli, conf_norm

# --- UI ---
cats  = ["Tümü"] + sorted(list(DFS.values())[0]['category'].unique().tolist())
diffs = ["Tümü", "easy", "medium", "hard"]

with gr.Blocks(title="Türkçe LLM Faithfulness Demo", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# Türkçe LLM Sadakat ve Epistemik Kalibrasyon Demosu")
    gr.Markdown("Veri setinden bir soru seçin - modelin chain-of-thought reasoning adimlarini gold evidence ile karsilastirin.")

    with gr.Row():
        dd_model = gr.Dropdown(choices=list(DFS.keys()), value=list(DFS.keys())[0], label="Model", scale=1)
        dd_cat   = gr.Dropdown(choices=cats,  value="Tümü", label="Kategori", scale=2)
        dd_diff  = gr.Dropdown(choices=diffs, value="Tümü", label="Zorluk",   scale=1)

    dd_soru = gr.Dropdown(label="Soru", choices=[])
    btn = gr.Button("Analiz Et", variant="primary", size="lg")

    with gr.Row():
        with gr.Column(scale=3):
            out_info      = gr.Markdown()
            out_reasoning = gr.Markdown()
            out_evidence  = gr.Markdown()

        with gr.Column(scale=2):
            out_faith_md = gr.Markdown()
            out_cal_md   = gr.Markdown()

            gr.Markdown("### Görsel Özet")
            with gr.Row():
                num_faith = gr.Number(label="Faithfulness",        precision=3)
                num_sem   = gr.Number(label="Semantik Benzerlik",  precision=3)
            with gr.Row():
                num_nli   = gr.Number(label="NLI Skoru",           precision=3)
                num_conf  = gr.Number(label="Güven (norm)",        precision=2)

    for trigger in [dd_model, dd_cat, dd_diff]:
        trigger.change(get_soru_listesi, inputs=[dd_model, dd_cat, dd_diff], outputs=dd_soru)

    btn.click(
        analiz_et,
        inputs=[dd_model, dd_cat, dd_diff, dd_soru],
        outputs=[out_info, out_reasoning, out_evidence, out_faith_md, out_cal_md,
                 num_faith, num_sem, num_nli, num_conf]
    )

    demo.load(get_soru_listesi, inputs=[dd_model, dd_cat, dd_diff], outputs=dd_soru)

demo.launch(share=True, debug=False)

/tmp/ipykernel_1171/2160865092.py:130: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Türkçe LLM Faithfulness Demo", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4056fb361193d76e8c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
